# MAUDE NLP Classifier — Exploratory Data Analysis

This notebook walks through:
1. Fetching sample records from the openFDA MAUDE API
2. Exploring the label distribution
3. Visualizing narrative text length and common terms
4. Running a quick baseline training run

In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.ingestion.openfda_client import fetch_maude_records
from src.preprocessing.text_cleaner import clean_dataframe
from src.model.classifier import build_pipeline, split_data, train_pipeline, evaluate

%matplotlib inline
sns.set_theme(style='whitegrid')

## 1. Fetch Data

In [ ]:
df_raw = fetch_maude_records(total_records=500)
print(f"Fetched {len(df_raw)} records")
df_raw.head()

## 2. Label Distribution

In [ ]:
label_counts = df_raw['severity_label'].value_counts()
print(label_counts)

fig, ax = plt.subplots(figsize=(8, 4))
label_counts.plot(kind='bar', ax=ax, color=['#d62728','#ff7f0e','#e6c619','#1f77b4','#7f7f7f'])
ax.set_title('MAUDE Adverse Event Severity Distribution')
ax.set_xlabel('Severity Label')
ax.set_ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 3. Narrative Text Length

In [ ]:
df_raw['text_length'] = df_raw['narrative_text'].str.len()

fig, ax = plt.subplots(figsize=(8, 4))
df_raw.groupby('severity_label')['text_length'].median().sort_values().plot(
    kind='barh', ax=ax)
ax.set_title('Median Narrative Length by Severity')
ax.set_xlabel('Median Characters')
plt.tight_layout()
plt.show()

## 4. Preprocess & Quick Baseline

In [ ]:
df = clean_dataframe(df_raw)
df = df[df['severity_label'] != 'UNKNOWN']
print(f"Clean records: {len(df)}")

X_train, X_test, y_train, y_test = split_data(df)
pipe = build_pipeline('logreg')
pipe = train_pipeline(pipe, X_train, y_train)
metrics = evaluate(pipe, X_test, y_test)

print(f"\nAccuracy: {metrics['accuracy']:.3f}")
print(f"Weighted F1: {metrics['f1_weighted']:.3f}")
print(metrics['classification_report'])

## 5. Top TF-IDF Features per Class

In [ ]:
import numpy as np

tfidf = pipe.named_steps['tfidf']
clf   = pipe.named_steps['clf']
feature_names = np.array(tfidf.get_feature_names_out())

if hasattr(clf, 'coef_'):
    for i, class_label in enumerate(pipe.classes_):
        top_idx = clf.coef_[i].argsort()[-15:]
        print(f"\n--- Top features for {class_label} ---")
        print(', '.join(feature_names[top_idx][::-1]))
else:
    print("Feature importance not available for this classifier type.")